In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Project paths
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data" / "raw"
REPORT_DIR = PROJECT_DIR / "reports"

print("CREDRESOLVE — PHASE 2: DATA QUALITY ANALYSIS")
print("=" * 55)
print("Project directory:", PROJECT_DIR)
print("Raw data directory exists:", DATA_DIR.exists())
print("Reports directory exists:", REPORT_DIR.exists())

# Load existing profiling reports
for filename in ["profile_summary.csv", "column_profile.csv"]:
    filepath = REPORT_DIR / filename
    if filepath.exists():
        print(f"\n--- {filename} ---")
        display(pd.read_csv(filepath).head(20))
    else:
        print(f"\nReport not found: {filename}")

CREDRESOLVE — PHASE 2: DATA QUALITY ANALYSIS
Project directory: C:\Users\kaspa\CredResolve-Data-Analyst-Assignment
Raw data directory exists: True
Reports directory exists: True

--- profile_summary.csv ---


,dataset,rows,columns,missing_cells,duplicate_rows,memory_mb,status,error
0,account_status_history,60000,8,0,0,31.40,SUCCESS,NaN
1,accounts,30000,11,455,0,15.62,SUCCESS,NaN
2,agent_sessions,15000,7,0,0,6.83,SUCCESS,NaN
3,agents,30000,8,0,0,15.56,SUCCESS,NaN
4,borrowers,30600,8,1509,600,14.61,SUCCESS,NaN
5,call_attempts,120000,9,2400,0,63.43,SUCCESS,NaN
6,call_dispositions,35000,8,0,0,18.21,SUCCESS,NaN
7,calls,91350,11,1827,1271,59.31,SUCCESS,NaN
8,campaigns,120,7,0,0,0.05,SUCCESS,NaN
9,complaints,8000,9,0,0,4.68,SUCCESS,NaN



--- column_profile.csv ---


,dataset,column,dtype,missing_count,missing_percent,unique_values
0,account_status_history,history_id,object,0,0.00,60000
1,account_status_history,account_id,object,0,0.00,25999
2,account_status_history,borrower_id,object,0,0.00,11916
3,account_status_history,event_at,object,0,0.00,59898
4,account_status_history,status,object,0,0.00,7
5,account_status_history,changed_by,object,0,0.00,101
6,account_status_history,source,object,0,0.00,5
7,account_status_history,recorded_at,object,0,0.00,59906
8,accounts,account_id,object,0,0.00,30000
9,accounts,borrower_id,object,455,1.52,10943


In [2]:
# 2. Missing-Value Analysis

column_profile = pd.read_csv(REPORT_DIR / "column_profile.csv")

missing_values = column_profile[
    column_profile["missing_count"] > 0
].sort_values("missing_count", ascending=False)

print("MISSING-VALUE SUMMARY")
print("=" * 50)

display(
    missing_values[
        ["dataset", "column", "missing_count", "missing_percent"]
    ].reset_index(drop=True)
)

print("\nTotal columns with missing values:", len(missing_values))
print(
    "Datasets affected:",
    missing_values["dataset"].nunique()
)

MISSING-VALUE SUMMARY


,dataset,column,missing_count,missing_percent
0,call_attempts,vendor_id,2400,2.00
1,calls,agent_id,1827,2.00
2,borrowers,email,895,2.92
3,borrowers,phone,614,2.01
4,accounts,borrower_id,455,1.52
5,payments,payment_reference,382,1.50
6,field_visits,scheduled_at,250,1.00



Total columns with missing values: 7
Datasets affected: 6


In [3]:
# 3. Duplicate-Record Investigation

profile_summary = pd.read_csv(REPORT_DIR / "profile_summary.csv")

print("DUPLICATE-RECORD SUMMARY")
print("=" * 50)

display(
    profile_summary[
        profile_summary["duplicate_rows"] > 0
    ][["dataset", "rows", "duplicate_rows"]]
    .reset_index(drop=True)
)

DUPLICATE-RECORD SUMMARY


,dataset,rows,duplicate_rows
0,borrowers,30600,600
1,calls,91350,1271
2,payments,25500,486
3,whatsapp_events,60600,600


In [4]:
# 4. Referential-Integrity Analysis

accounts = pd.read_csv(DATA_DIR / "accounts.csv")
history = pd.read_csv(DATA_DIR / "account_status_history.csv")

missing_accounts = accounts[
    accounts["borrower_id"].isna()
].copy()

history_account_ids = set(
    history["account_id"].dropna().astype(str)
)

missing_accounts["found_in_history"] = (
    missing_accounts["account_id"].astype(str)
    .isin(history_account_ids)
)

matched = int(missing_accounts["found_in_history"].sum())
unmatched = len(missing_accounts) - matched

referential_summary = pd.DataFrame({
    "Metric": [
        "Total accounts",
        "Accounts with missing borrower_id",
        "Missing-borrower accounts found in status history",
        "Missing-borrower accounts not found in status history"
    ],
    "Count": [
        len(accounts),
        len(missing_accounts),
        matched,
        unmatched
    ]
})

display(referential_summary)

,Metric,Count
0,Total accounts,30000
1,Accounts with missing borrower_id,455
2,Missing-borrower accounts found in status history,414
3,Missing-borrower accounts not found in status ...,41


In [5]:
# 5. Timestamp and Timezone Quality Analysis

calls = pd.read_csv(DATA_DIR / "calls.csv")

print("CALLS DATASET")
print("=" * 50)
print("Rows:", len(calls))
print("\nAvailable columns:")
print(calls.columns.tolist())

timestamp_columns = [
    col for col in calls.columns
    if any(word in col.lower() for word in ["time", "date", "timestamp"])
]

timezone_columns = [
    col for col in calls.columns
    if "timezone" in col.lower() or "tz" == col.lower()
]

print("\nTimestamp/date columns:", timestamp_columns)
print("Timezone columns:", timezone_columns)

for col in timestamp_columns:
    print(f"\n{col}:")
    print("Missing values:", calls[col].isna().sum())
    print("Unparseable values:", pd.to_datetime(
        calls[col], errors="coerce", utc=True
    ).isna().sum() - calls[col].isna().sum())

for col in timezone_columns:
    print(f"\nTimezone distribution — {col}:")
    display(calls[col].value_counts(dropna=False).to_frame("count"))

CALLS DATASET
Rows: 91350

Available columns:
['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone']

Timestamp/date columns: ['timezone']
Timezone columns: ['timezone']

timezone:
Missing values: 0
Unparseable values: 91350

Timezone distribution — timezone:


C:\Users\kaspa\AppData\Local\Temp\ipykernel_23936\834269894.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  print("Unparseable values:", pd.to_datetime(


,count
timezone,
Asia/Kolkata,30485
Asia/Dubai,30464
UTC,30401


In [6]:
# 5. Corrected Timestamp and Timezone Analysis

calls = pd.read_csv(DATA_DIR / "calls.csv")

calls["event_at_parsed"] = pd.to_datetime(
    calls["event_at"], errors="coerce", utc=True
)

print("CALL TIMESTAMP QUALITY")
print("=" * 50)
print("Total call records:", len(calls))
print("Missing event_at:", calls["event_at"].isna().sum())
print(
    "Invalid event_at:",
    (
        calls["event_at"].notna()
        & calls["event_at_parsed"].isna()
    ).sum()
)

print("\nTimezone distribution:")
display(calls["timezone"].value_counts(dropna=False).to_frame("records"))

print("\nParsed timestamp range:")
print("Earliest:", calls["event_at_parsed"].min())
print("Latest:", calls["event_at_parsed"].max())

CALL TIMESTAMP QUALITY
Total call records: 91350
Missing event_at: 0
Invalid event_at: 0

Timezone distribution:


,records
timezone,
Asia/Kolkata,30485
Asia/Dubai,30464
UTC,30401



Parsed timestamp range:
Earliest: 2025-12-29 06:52:37+00:00
Latest: 2026-08-12 15:43:05+00:00


In [7]:
# 6. Business Impact and Recommendations

findings = pd.DataFrame([
    {
        "Issue": "Missing borrower IDs",
        "Evidence": "455 accounts (1.52%)",
        "Business Impact": "Account-to-borrower mapping and reporting may be incomplete",
        "Recommendation": "Reconcile with trusted source records before updating"
    },
    {
        "Issue": "Missing agent IDs",
        "Evidence": "1,827 call records (2.00%)",
        "Business Impact": "Agent performance attribution may be inaccurate",
        "Recommendation": "Validate against agent and telephony mapping tables"
    },
    {
        "Issue": "Missing vendor IDs",
        "Evidence": "2,400 call attempts (2.00%)",
        "Business Impact": "Vendor-level operational metrics may be incomplete",
        "Recommendation": "Reconcile with vendor logs and source identifiers"
    },
    {
        "Issue": "Missing borrower contact details",
        "Evidence": "895 missing emails; 614 missing phone numbers",
        "Business Impact": "Contact-channel analysis may be incomplete",
        "Recommendation": "Validate contact details through approved processes"
    },
    {
        "Issue": "Duplicate rows",
        "Evidence": "Reported in borrowers, calls, payments and WhatsApp events",
        "Business Impact": "Unverified duplicates may inflate counts and totals",
        "Recommendation": "Investigate business keys before deduplication"
    },
    {
        "Issue": "Mixed timezones",
        "Evidence": "Asia/Kolkata, Asia/Dubai and UTC in call records",
        "Business Impact": "Time-based comparisons may be misleading if timestamps are mishandled",
        "Recommendation": "Standardize event timestamps to UTC while retaining original timezone"
    },
    {
        "Issue": "Missing payment references",
        "Evidence": "382 payment records (1.50%)",
        "Business Impact": "Payment reconciliation may require additional investigation",
        "Recommendation": "Reconcile against trusted payment-provider records"
    },
    {
        "Issue": "Missing field-visit schedules",
        "Evidence": "250 records (1.00%)",
        "Business Impact": "Visit scheduling and completion analysis may be affected",
        "Recommendation": "Validate against field-operations records"
    }
])

display(findings)

print("\n7. CONCLUSION")
print("=" * 50)
print("""
Profiling identified missing values, duplicate rows and mixed timezones
across multiple datasets. These issues can affect operational reporting,
reconciliation and performance analysis.

Recommended next steps:
1. Resolve missing identifiers using trusted source records.
2. Investigate duplicates before removing any records.
3. Validate relationships between datasets.
4. Standardize timestamp handling and preserve original timezone metadata.
5. Document all cleaning rules and validate results before downstream analysis.

Limitation: These findings identify data-quality risks; they do not prove
that every flagged record is incorrect or quantify actual financial loss.
""")

,Issue,Evidence,Business Impact,Recommendation
0,Missing borrower IDs,455 accounts (1.52%),Account-to-borrower mapping and reporting may ...,Reconcile with trusted source records before u...
1,Missing agent IDs,"1,827 call records (2.00%)",Agent performance attribution may be inaccurate,Validate against agent and telephony mapping t...
2,Missing vendor IDs,"2,400 call attempts (2.00%)",Vendor-level operational metrics may be incomp...,Reconcile with vendor logs and source identifiers
3,Missing borrower contact details,895 missing emails; 614 missing phone numbers,Contact-channel analysis may be incomplete,Validate contact details through approved proc...
4,Duplicate rows,"Reported in borrowers, calls, payments and Wha...",Unverified duplicates may inflate counts and t...,Investigate business keys before deduplication
5,Mixed timezones,"Asia/Kolkata, Asia/Dubai and UTC in call records",Time-based comparisons may be misleading if ti...,Standardize event timestamps to UTC while reta...
6,Missing payment references,382 payment records (1.50%),Payment reconciliation may require additional ...,Reconcile against trusted payment-provider rec...
7,Missing field-visit schedules,250 records (1.00%),Visit scheduling and completion analysis may b...,Validate against field-operations records



7. CONCLUSION

Profiling identified missing values, duplicate rows and mixed timezones
across multiple datasets. These issues can affect operational reporting,
reconciliation and performance analysis.

Recommended next steps:
1. Resolve missing identifiers using trusted source records.
2. Investigate duplicates before removing any records.
3. Validate relationships between datasets.
4. Standardize timestamp handling and preserve original timezone metadata.
5. Document all cleaning rules and validate results before downstream analysis.

Limitation: These findings identify data-quality risks; they do not prove
that every flagged record is incorrect or quantify actual financial loss.



In [8]:
# PROJECT AUDIT — Identify existing deliverables

from pathlib import Path
import pandas as pd

folders = ["notebooks", "sql", "reports", "dashboard", "docs", "src", "tests"]

inventory = []

for folder in folders:
    folder_path = PROJECT_DIR / folder

    if folder_path.exists():
        for file in folder_path.rglob("*"):
            if file.is_file() and ".venv" not in file.parts:
                inventory.append({
                    "folder": folder,
                    "file": str(file.relative_to(PROJECT_DIR)),
                    "size_kb": round(file.stat().st_size / 1024, 2)
                })

inventory_df = pd.DataFrame(inventory)

if not inventory_df.empty:
    display(
        inventory_df.sort_values(["folder", "file"])
        .reset_index(drop=True)
    )

    print("\nTOTAL FILES:", len(inventory_df))
else:
    print("No files found. Check PROJECT_DIR:", PROJECT_DIR)

,folder,file,size_kb
0,notebooks,notebooks\.ipynb_checkpoints\02_data_quality_a...,48.94
1,notebooks,notebooks\02_data_quality_analysis.ipynb,48.94
2,reports,reports\account_borrower_quality_summary.csv,0.18
3,reports,reports\account_borrower_reference_summary.csv,0.12
4,reports,reports\account_status_summary.csv,0.14
5,reports,reports\accounts_with_invalid_borrower_referen...,288.02
6,reports,reports\accounts_without_status_history.csv,467.85
7,reports,reports\borrower_record_frequencies.csv,193.65
8,reports,reports\borrowers_with_missing_contact_details...,164.72
9,reports,reports\call_attempts_missing_vendor_ids.csv,257.82



TOTAL FILES: 32


### Monthly Recovery Trend — Preliminary Findings

The analysis of successful payment records shows fluctuations in recorded payment amounts during 2026.

- **May 2026:** +5.20% month-on-month.
- **June 2026:** -4.60% month-on-month.
- **July 2026:** +6.48% month-on-month.
- **August 2026:** -74.84% month-on-month. The dataset ends on August 8, so this comparison is incomplete.

**Conclusion:** The available evidence does not establish whether the claimed 11% month-on-month improvement is true. These figures represent recorded successful payment amounts, not a fully validated recovery rate. Further investigation of duplicate transactions, portfolio mix, and data completeness is required before drawing a final conclusion.